# Khối C — Fine-tune VietOCR `vgg_transformer` đọc vùng chữ đơn thuốc

Tái lập đúng lần train cho `code/weights/vietocr_finetuned.pth` (plan.md §5.3, phương án **C3**), đồng thời chấm
VietOCR **pretrained (C2)** trên cùng vùng cắt test → 2 dòng của bảng E-C1.
Notebook **tự chứa**: mọi code nằm trong các cell dưới, không clone repo nào. Bước cắt vùng cùng logic với
`code/tools/to_vietocr.py` (đã đối chiếu: 6.086/6.086 vùng test khớp đường dẫn, chữ và lớp).

| Tham số | Giá trị | Lý do |
| :--- | :--- | :--- |
| Mô hình | `vgg_transformer` pretrained (vocr.vn), `vietocr==0.3.13` | |
| Dữ liệu | mọi vùng chữ có nhãn (6 lớp), cắt theo box chuẩn, đệm 2 px, không phóng | ~28,8k train / 6,3k val / 6,1k test |
| Chữ | Unicode NFC + gộp khoảng trắng; bỏ vùng có ký tự ngoài vocab (thực tế: 0) | khớp chuẩn hóa của CER |
| `iters` / `batch` / `max_lr` | 20 000 / 32 / 3e-4 (one-cycle) | |
| `valid_every` | 2 000 — chỉ export trọng số ở các lần val | chọn theo đúng nguyên dòng trên **val** |
| `image_max_width` | 1024 (mặc định 512) | ~5 % vùng (dòng chẩn đoán) rộng hơn 512 px ở cao 32 |
| `seed` | 42 | |
| GPU | 1 × T4 (`CUDA_VISIBLE_DEVICES=0`) | |

Kết quả lần chạy gốc (test, 6.086 vùng):

| | CER | WER | Đúng nguyên dòng |
| :--- | :---: | :---: | :---: |
| C2 pretrained | 0,0864 | 0,1097 | 0,8156 |
| **C3 fine-tune** | **0,0329** | **0,0437** | **0,9229** |

Thời gian ≈ 3–5 giờ (20k iter).

## Cách chạy trên Kaggle

1. **Create → New Notebook → File → Import Notebook**, chọn file này.
2. **Add Input** → tìm `vaipepill2022` (của `tommyngx`) → **Add**. Không cần upload dữ liệu: notebook đọc thẳng
   `public_train/prescription/{image,label}` và chia split theo danh sách nhúng sẵn ở cell *Dữ liệu*.
3. **Session options**: Accelerator **GPU T4 x2**, Internet **On** (để `pip install` và tải trọng số pretrained).
4. Chạy thử: đặt `QUICK = True` ở cell *Tham số* → **Run All** (~10–15 phút) để bắt lỗi môi trường.
5. Chạy thật: `QUICK = False` → **Save Version → Save & Run All (Commit)**, *Advanced*: *Always save output*.
6. Lấy kết quả ở tab **Output**: `output/results.zip`. Đặt trọng số vào `code/weights/` của repo (tên ở cuối notebook).

Môi trường lúc chạy gốc (11/11/2026): image Kaggle Python 3.13.16, torch 2.11.0+cu130, 2 × Tesla T4.
Huấn luyện trên GPU không tất định hoàn toàn, nên chạy lại có thể lệch nhẹ ở chữ số thập phân thứ 3–4.

## Tham số

In [ ]:
QUICK = False            # True: 400 iter, val mỗi 200 — kiểm tra môi trường
ITERS = 400 if QUICK else 20000
VALID_EVERY = 200 if QUICK else 2000
BATCH, MAX_WIDTH, LR, SEED, PAD = 32, 1024, 3e-4, 42, 2
WORK = '/kaggle/working'
OUT_DIR = f'{WORK}/output'

## Cài đặt

In [ ]:
# vietocr 0.3.13 ghim pillow/einops/imgaug cũ → cài không kèm phụ thuộc, rồi cài phụ thuộc thật sự cần
!pip install -q --no-deps vietocr==0.3.13
!pip install -q einops lmdb prefetch_generator albumentations gdown six
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'   # torch>=2.6 mặc định weights_only=True
import sys, torch
print('python', sys.version.split()[0], '| torch', torch.__version__)
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, 'Chưa bật GPU (Accelerator = GPU T4 x2)'

## Config VietOCR

`base.yml` + `vgg-transformer.yml` của vocr.vn, lưu sẵn để không phụ thuộc lúc tải. Cùng file với `code/configs/vietocr_vgg_transformer.yml` (bản repo đổi `device: cpu`, `image_max_width: 1024` cho suy luận local; ở đây script ghi đè 2 giá trị này).

In [ ]:
%%writefile /kaggle/working/vgg_transformer.yml
# VietOCR vgg_transformer config (base.yml + vgg-transformer.yml from vocr.vn),
# stored locally so training does not depend on the config download.
vocab: 'aAàÀảẢãÃáÁạẠăĂằẰẳẲẵẴắẮặẶâÂầẦẩẨẫẪấẤậẬbBcCdDđĐeEèÈẻẺẽẼéÉẹẸêÊềỀểỂễỄếẾệỆfFgGhHiIìÌỉỈĩĨíÍịỊjJkKlLmMnNoOòÒỏỎõÕóÓọỌôÔồỒổỔỗỖốỐộỘơƠờỜởỞỡỠớỚợỢpPqQrRsStTuUùÙủỦũŨúÚụỤưƯừỪửỬữỮứỨựỰvVwWxXyYỳỲỷỶỹỸýÝỵỴzZ0123456789!"#$%&''()*+,-./:;<=>?@[\]^_`{|}~ '
device: cuda:0
seq_modeling: transformer
transformer:
  d_model: 256
  nhead: 8
  num_encoder_layers: 6
  num_decoder_layers: 6
  dim_feedforward: 2048
  max_seq_length: 1024
  pos_dropout: 0.1
  trans_dropout: 0.1
optimizer:
  max_lr: 0.0003
  pct_start: 0.1
trainer:
  batch_size: 32
  print_every: 200
  valid_every: 4000
  iters: 100000
  export: ./weights/transformerocr.pth
  checkpoint: ./checkpoint/transformerocr_checkpoint.pth
  log: ./train.log
  metrics: null
dataset:
  name: data
  data_root: ./img/
  train_annotation: annotation_train.txt
  valid_annotation: annotation_val_small.txt
  image_height: 32
  image_min_width: 32
  image_max_width: 512
dataloader:
  num_workers: 3
  pin_memory: true
aug:
  image_aug: true
  masked_language_model: true
predictor:
  beamsearch: false
quiet: false
pretrain: https://vocr.vn/data/vietocr/vgg_transformer.pth
weights: https://vocr.vn/data/vietocr/vgg_transformer.pth
backbone: vgg19_bn
cnn:
  pretrained: true
  ss:
  - - 2
    - 2
  - - 2
    - 2
  - - 2
    - 1
  - - 2
    - 1
  - - 1
    - 1
  ks:
  - - 2
    - 2
  - - 2
    - 2
  - - 2
    - 1
  - - 2
    - 1
  - - 1
    - 1
  hidden: 256

## Dữ liệu

In [ ]:
# ---- Dữ liệu: tìm public_train của vaipepill2022, chia split theo danh sách cố định ----
# Bộ chia = data/1_vaipe/split_manifest.csv của repo (gom theo bệnh nhân, seed 42): train/valid/test = 820/176/176.
# Mọi ảnh là VAIPE_P_TRAIN_<số>; VAIPE_P_TRAIN_1172 bị loại (trùng byte với VAIPE_P_TEST_32).
import collections, json, os, shutil, sys, unicodedata
from pathlib import Path

SPLIT_IDS = {"train": [0, 3, 4, 6, 7, 8, 9, 12, 13, 14, 16, 18, 22, 23, 24, 25, 26, 30, 31, 32, 34, 35, 37, 38, 39, 40, 41, 42, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 58, 59, 60, 61, 62, 63, 66, 67, 68, 69, 70, 72, 73, 75, 76, 77, 79, 80, 81, 83, 84, 85, 86, 87, 90, 91, 92, 95, 96, 98, 99, 101, 102, 103, 104, 107, 108, 109, 110, 111, 112, 114, 115, 116, 118, 119, 120, 121, 123, 124, 125, 126, 127, 128, 130, 131, 132, 135, 136, 138, 140, 141, 142, 143, 145, 147, 149, 152, 154, 159, 160, 162, 163, 166, 167, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 183, 184, 185, 186, 187, 188, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 201, 206, 208, 209, 211, 212, 213, 214, 218, 220, 221, 225, 226, 230, 235, 237, 238, 239, 240, 243, 244, 245, 246, 247, 249, 250, 251, 253, 254, 255, 256, 257, 259, 260, 263, 266, 267, 268, 270, 271, 272, 273, 274, 275, 276, 292, 294, 295, 296, 297, 300, 302, 303, 304, 305, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 325, 326, 327, 328, 329, 331, 333, 335, 336, 337, 338, 339, 341, 343, 344, 345, 347, 348, 349, 350, 351, 352, 354, 355, 356, 358, 359, 363, 365, 367, 368, 369, 371, 374, 375, 377, 379, 380, 382, 384, 385, 386, 387, 388, 391, 392, 394, 395, 396, 400, 401, 407, 408, 409, 410, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 430, 432, 433, 435, 436, 437, 440, 443, 444, 445, 447, 449, 451, 452, 455, 456, 457, 458, 463, 465, 466, 467, 468, 469, 472, 473, 474, 475, 476, 477, 479, 482, 483, 486, 489, 491, 492, 493, 494, 495, 496, 497, 498, 500, 503, 504, 505, 506, 508, 510, 512, 513, 514, 515, 516, 518, 519, 521, 523, 524, 525, 526, 527, 528, 529, 532, 533, 536, 538, 539, 540, 541, 542, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 559, 560, 561, 562, 563, 564, 565, 566, 568, 569, 570, 571, 573, 576, 577, 579, 581, 582, 584, 587, 588, 589, 590, 591, 592, 595, 596, 598, 599, 600, 601, 603, 604, 608, 609, 610, 611, 612, 613, 614, 615, 616, 617, 618, 620, 622, 623, 625, 627, 628, 629, 631, 633, 635, 636, 637, 638, 639, 640, 642, 643, 644, 647, 648, 649, 652, 653, 656, 657, 659, 660, 661, 662, 663, 664, 665, 667, 668, 669, 670, 671, 672, 673, 674, 676, 677, 678, 681, 682, 683, 685, 686, 688, 689, 693, 694, 695, 696, 697, 698, 699, 700, 701, 702, 703, 704, 706, 707, 708, 709, 710, 713, 716, 717, 718, 719, 721, 723, 724, 725, 726, 727, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 742, 743, 744, 745, 746, 747, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 760, 761, 762, 763, 764, 765, 768, 770, 771, 774, 776, 777, 778, 779, 781, 782, 783, 785, 786, 787, 790, 791, 794, 795, 796, 799, 800, 801, 803, 804, 805, 806, 807, 808, 810, 811, 812, 813, 814, 815, 816, 817, 819, 820, 821, 822, 824, 825, 826, 827, 829, 830, 831, 832, 833, 836, 838, 839, 842, 844, 845, 846, 847, 848, 849, 850, 852, 854, 856, 857, 858, 859, 860, 861, 864, 865, 866, 867, 869, 870, 871, 872, 873, 874, 875, 877, 878, 879, 882, 883, 884, 885, 886, 887, 890, 891, 893, 895, 897, 898, 899, 900, 901, 903, 905, 906, 909, 910, 914, 916, 917, 918, 919, 920, 921, 922, 924, 925, 926, 930, 931, 932, 933, 934, 935, 936, 937, 938, 939, 941, 942, 943, 944, 945, 946, 948, 949, 950, 951, 952, 953, 957, 958, 959, 962, 964, 966, 967, 968, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 979, 981, 982, 985, 986, 987, 988, 989, 990, 991, 992, 993, 994, 996, 997, 998, 999, 1000, 1001, 1003, 1006, 1007, 1009, 1011, 1012, 1013, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1024, 1025, 1026, 1028, 1029, 1031, 1033, 1034, 1036, 1037, 1038, 1039, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1050, 1051, 1053, 1054, 1056, 1057, 1058, 1059, 1060, 1061, 1063, 1064, 1065, 1066, 1067, 1070, 1071, 1072, 1073, 1076, 1077, 1078, 1079, 1080, 1081, 1083, 1084, 1086, 1087, 1089, 1090, 1091, 1092, 1095, 1096, 1097, 1098, 1099, 1100, 1101, 1102, 1103, 1104, 1107, 1111, 1112, 1113, 1114, 1116, 1118, 1119, 1120, 1121, 1122, 1123, 1124, 1125, 1126, 1127, 1129, 1131, 1133, 1134, 1136, 1137, 1138, 1139, 1140, 1141, 1142, 1143, 1144, 1146, 1147, 1148, 1149, 1150, 1152, 1153, 1155, 1156, 1157, 1158, 1159, 1160, 1162, 1164, 1165, 1166, 1167, 1168, 1170, 1171], "valid": [2, 10, 11, 17, 27, 28, 44, 64, 65, 74, 88, 100, 106, 113, 122, 133, 134, 144, 146, 148, 150, 151, 153, 155, 156, 157, 158, 161, 164, 165, 168, 169, 200, 202, 210, 215, 219, 224, 229, 231, 233, 234, 236, 241, 248, 252, 258, 261, 265, 277, 280, 298, 299, 301, 306, 307, 308, 321, 322, 332, 340, 342, 346, 360, 361, 362, 364, 373, 376, 381, 389, 393, 399, 406, 429, 431, 434, 438, 439, 459, 462, 471, 478, 480, 484, 501, 507, 531, 535, 537, 543, 544, 558, 580, 585, 586, 593, 621, 624, 630, 641, 654, 655, 658, 675, 680, 684, 687, 705, 711, 714, 715, 720, 730, 759, 775, 780, 788, 793, 797, 798, 802, 823, 828, 834, 835, 837, 840, 841, 851, 862, 889, 892, 894, 902, 907, 912, 913, 915, 927, 928, 929, 940, 954, 955, 956, 960, 961, 963, 983, 984, 1002, 1004, 1008, 1010, 1014, 1023, 1040, 1052, 1062, 1068, 1069, 1074, 1088, 1093, 1094, 1105, 1106, 1108, 1110, 1115, 1117, 1130, 1145, 1151, 1169], "test": [1, 5, 15, 19, 20, 21, 29, 33, 36, 43, 56, 57, 71, 78, 82, 89, 93, 94, 97, 105, 117, 129, 137, 139, 180, 182, 189, 203, 204, 205, 207, 216, 217, 222, 223, 227, 228, 232, 242, 262, 264, 269, 278, 279, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 293, 323, 324, 330, 334, 353, 357, 366, 370, 372, 378, 383, 390, 397, 398, 402, 403, 404, 405, 411, 441, 442, 446, 448, 450, 453, 454, 460, 461, 464, 470, 481, 485, 487, 488, 490, 499, 502, 509, 511, 517, 520, 522, 530, 534, 567, 572, 574, 575, 578, 583, 594, 597, 602, 605, 606, 607, 619, 626, 632, 634, 645, 646, 650, 651, 666, 679, 690, 691, 692, 712, 722, 728, 729, 748, 766, 767, 769, 772, 773, 784, 789, 792, 809, 818, 843, 853, 855, 863, 868, 876, 880, 881, 888, 896, 904, 908, 911, 923, 947, 965, 980, 995, 1005, 1027, 1030, 1032, 1035, 1048, 1049, 1055, 1075, 1082, 1085, 1109, 1128, 1132, 1135, 1154, 1161, 1163]}
SPLITS = {s: sorted(f'VAIPE_P_TRAIN_{i}' for i in ids) for s, ids in SPLIT_IDS.items()}
CLASSES = ['drugname', 'quantity', 'usage', 'diagnose', 'date', 'other']   # thứ tự = class_id YOLO
EXPECTED = {  # số vùng chữ theo lớp — phải khớp, nếu không là sai dataset/split
    'train': {'images': 820, 'drugname': 1995, 'quantity': 1995, 'usage': 2809, 'diagnose': 1109, 'date': 820, 'other': 20108},
    'valid': {'images': 176, 'drugname': 405, 'quantity': 405, 'usage': 583, 'diagnose': 287, 'date': 176, 'other': 4402},
    'test':  {'images': 176, 'drugname': 443, 'quantity': 443, 'usage': 602, 'diagnose': 253, 'date': 176, 'other': 4169},
}


def find_public_train(search='/kaggle/input', max_depth=6):
    queue = [(Path(search), 0)]
    while queue:
        d, depth = queue.pop(0)
        if (d / 'prescription' / 'label').is_dir() and d.name == 'public_train':
            return d
        if depth < max_depth:
            try:
                queue += [(p, depth + 1) for p in sorted(d.iterdir()) if p.is_dir()
                          and p.name not in ('image', 'label', 'pill')]
            except OSError:
                pass
    return None


DATA_ROOT = Path(os.environ.get('DATA_ROOT') or find_public_train() or '')
LABEL_DIR, IMAGE_DIR = DATA_ROOT / 'prescription' / 'label', DATA_ROOT / 'prescription' / 'image'
assert LABEL_DIR.is_dir(), 'Không thấy public_train/prescription/label — Add Input dataset tommyngx/vaipepill2022'
on_disk = {p.stem for p in LABEL_DIR.glob('*.json')}
listed = {s for v in SPLITS.values() for s in v}
assert listed <= on_disk, f'Thiếu nhãn: {sorted(listed - on_disk)[:5]}'
print('DATA_ROOT =', DATA_ROOT)
print('nhãn không thuộc split nào (bỏ qua):', sorted(on_disk - listed))


def image_path(stem):
    for ext in ('.png', '.jpg', '.jpeg'):
        if (IMAGE_DIR / (stem + ext)).exists():
            return IMAGE_DIR / (stem + ext)
    raise FileNotFoundError(stem)


def read_label(stem):
    with open(LABEL_DIR / f'{stem}.json', encoding='utf-8') as f:
        return json.load(f)


def clean_text(text):
    """NFC + gộp khoảng trắng (= code/src/vaipe/dataset.py::clean_text)."""
    return ' '.join(unicodedata.normalize('NFC', str(text)).split())


def clip_box(box, w, h):
    """xyxy kẹp vào ảnh, làm tròn int (= code/src/common/boxes.py::clip_box)."""
    x1, y1, x2, y2 = (float(v) for v in box)
    x1, x2 = sorted((min(max(x1, 0), w), min(max(x2, 0), w)))
    y1, y2 = sorted((min(max(y1, 0), h), min(max(y2, 0), h)))
    return int(round(x1)), int(round(y1)), int(round(x2)), int(round(y2))


for split, stems in SPLITS.items():
    cnt = collections.Counter(a['label'] for s in stems for a in read_label(s))
    got = {'images': len(stems), **{c: cnt[c] for c in CLASSES}}
    print(f'{split:5s}:', got)
    assert got == EXPECTED[split], f'{split} lệch: {got} != {EXPECTED[split]}'

## Cắt vùng chữ

Định dạng VietOCR: `img/<split>/<ảnh>_<k>.png<TAB><chữ>`, `k` = thứ tự vùng trong file nhãn.
Kèm `<split>_meta.tsv` (đường dẫn, lớp, chữ) để tính CER theo lớp.

In [ ]:
import cv2, yaml

OCR_DIR = Path(WORK) / 'dataset' / 'vietocr'
if OCR_DIR.exists():
    shutil.rmtree(OCR_DIR)
vocab = set(yaml.safe_load(open(f'{WORK}/vgg_transformer.yml', encoding='utf-8'))['vocab'])
stats = {}
for split, folder in (('train', 'train'), ('valid', 'val'), ('test', 'test')):
    (OCR_DIR / 'img' / folder).mkdir(parents=True)
    lines, meta = [], []
    per_class, skip, bad = collections.Counter(), collections.Counter(), collections.Counter()
    for stem in SPLITS[split]:
        img = cv2.imread(str(image_path(stem)))
        h, w = img.shape[:2]
        for k, a in enumerate(read_label(stem)):
            label = a.get('label')
            if label not in CLASSES:
                continue
            text = clean_text(a.get('text', ''))
            x1, y1, x2, y2 = clip_box(a['box'], w, h)
            if not text:
                skip['empty_text'] += 1; continue
            if x2 - x1 < 2 or y2 - y1 < 2:
                skip['degenerate_box'] += 1; continue
            oov = [c for c in text if c not in vocab]
            if oov:
                bad.update(oov); skip['char_not_in_vocab'] += 1; continue
            x1, y1, x2, y2 = max(x1 - PAD, 0), max(y1 - PAD, 0), min(x2 + PAD, w), min(y2 + PAD, h)
            rel = f'img/{folder}/{stem}_{k:03d}.png'
            cv2.imwrite(str(OCR_DIR / rel), img[y1:y2, x1:x2])
            lines.append(f'{rel}\t{text}')
            meta.append(f'{rel}\t{label}\t{text}')
            per_class[label] += 1
    lines.sort(); meta.sort()
    (OCR_DIR / f'{folder}.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    (OCR_DIR / f'{folder}_meta.tsv').write_text('path\tlabel\ttext\n' + '\n'.join(meta) + '\n', encoding='utf-8')
    stats[folder] = {'images': len(SPLITS[split]), 'crops': len(lines),
                     'crops_per_class': {c: per_class[c] for c in CLASSES},
                     'skipped': dict(skip), 'chars_not_in_vocab': dict(bad)}
    print(folder, stats[folder])
    assert {c: per_class[c] for c in CLASSES} == {c: EXPECTED[split][c] for c in CLASSES} and not skip
json.dump({'classes': CLASSES, 'pad': PAD, **stats}, open(OCR_DIR / 'stats.json', 'w', encoding='utf-8'),
          indent=1, ensure_ascii=False)

## Fine-tune + chấm điểm

Chạy dưới dạng script (DataLoader của VietOCR cần `fork`, và tách khỏi kernel cho an toàn khi treo máy).
Trình tự: chấm C2 pretrained trên test → fine-tune 20k iter (val mỗi 2k, export khi tốt hơn) → chấm C3 trên test.

In [ ]:
%%writefile /kaggle/working/vietocr_ft.py
"""Fine-tune VietOCR vgg_transformer trên vùng chữ đơn thuốc (plan.md §5.3, phương án C3)
và chấm pretrained (C2) + fine-tune (C3) trên vùng cắt test theo box chuẩn (E-C1)."""
import argparse, json, os, random, shutil, sys, time, types, unicodedata
from pathlib import Path

# vietocr import imgaug ở mức module nhưng chỉ dùng albumentations; imgaug 0.4 hỏng với NumPy 2 → stub
try:
    import imgaug  # noqa: F401
except Exception:
    _stub = types.ModuleType('imgaug')
    _stub.augmenters = types.ModuleType('imgaug.augmenters')
    sys.modules['imgaug'] = _stub
    sys.modules['imgaug.augmenters'] = _stub.augmenters

import numpy as np

# vietocr 0.3.13 đọc lmdb bằng np.fromstring(bytes), NumPy mới không nhận → vá
_np_fromstring = np.fromstring


def _fromstring(string, dtype=float, count=-1, *, sep='', **kw):
    if sep == '' and isinstance(string, (bytes, bytearray, memoryview)):
        return np.frombuffer(string, dtype=dtype, count=count).copy()
    return _np_fromstring(string, dtype=dtype, count=count, sep=sep, **kw)


np.fromstring = _fromstring

import torch
from PIL import Image
from vietocr.model.trainer import Trainer
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

PRETRAINED_URL = 'https://vocr.vn/data/vietocr/vgg_transformer.pth'


# ---------------- độ đo: CER / WER / đúng nguyên dòng, toàn tập + theo lớp ----------------
def norm(s):
    return ' '.join(unicodedata.normalize('NFC', str(s)).split())


def edit_distance(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def _score(pairs):
    ce = ct = we = wt = exact = 0
    for t, p in pairs:
        ce += edit_distance(t, p); ct += len(t)
        tw, pw = t.split(), p.split()
        we += edit_distance(tw, pw); wt += len(tw)
        exact += t == p
    n = len(pairs)
    return {'lines': n, 'cer': ce / max(ct, 1), 'wer': we / max(wt, 1), 'acc_full_sequence': exact / max(n, 1)}


def evaluate(truths, preds, labels=None):
    pairs = [(norm(t), norm(p)) for t, p in zip(truths, preds)]
    result = {'all': _score(pairs)}
    if labels is not None:
        groups = {}
        for lab, pair in zip(labels, pairs):
            groups.setdefault(lab, []).append(pair)
        result['per_class'] = {lab: _score(g) for lab, g in sorted(groups.items())}
    return result


def format_table(result):
    rows = [('all', result['all'])] + list(result.get('per_class', {}).items())
    out = [f'{"class":10s} {"lines":>6s} {"CER":>7s} {"WER":>7s} {"acc":>7s}']
    for name, r in rows:
        out.append(f'{name:10s} {r["lines"]:6d} {r["cer"]:7.4f} {r["wer"]:7.4f} {r["acc_full_sequence"]:7.4f}')
    return '\n'.join(out)


# ---------------- chấm một model trên vùng cắt của một split ----------------
def read_split(data_root, split):
    rows = []
    with open(Path(data_root) / f'{split}_meta.tsv', encoding='utf-8') as f:
        next(f)
        for line in f:
            line = line.rstrip('\n')
            if line:
                rel, label, text = line.split('\t', 2)
                rows.append((rel, text, label))
    return rows


def predict_all(predictor, images, batch=64):
    order = sorted(range(len(images)), key=lambda i: images[i].size[0] / images[i].size[1])
    preds = [None] * len(images)
    for s in range(0, len(order), batch):
        idx = order[s:s + batch]
        chunk = [images[i] for i in idx]
        try:
            out = predictor.predict_batch(chunk)
        except Exception as e:
            print(f'predict_batch failed ({e!r}), predicting one by one')
            out = [predictor.predict(im) for im in chunk]
        for i, p in zip(idx, out):
            preds[i] = p
    return preds


def evaluate_split(config, data_root, split, out_dir, name, batch=64):
    rows = read_split(data_root, split)
    images = [Image.open(Path(data_root) / rel).convert('RGB') for rel, _, _ in rows]
    predictor = Predictor(config)
    cuda = str(config['device']).startswith('cuda')
    if cuda:
        torch.cuda.synchronize()
    t0 = time.time()
    preds = predict_all(predictor, images, batch)
    if cuda:
        torch.cuda.synchronize()
    seconds = time.time() - t0
    result = evaluate([t for _, t, _ in rows], preds, [l for _, _, l in rows])
    result.update({'name': name, 'split': split, 'weights': str(config['weights']),
                   'device': str(config['device']), 'image_max_width': config['dataset']['image_max_width'],
                   'seconds_per_crop': seconds / max(len(rows), 1)})
    out_dir = Path(out_dir)
    json.dump(result, open(out_dir / f'{name}_metrics.json', 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
    with open(out_dir / f'{name}_predictions.tsv', 'w', encoding='utf-8') as f:
        f.write('image\tlabel\tground_truth\tprediction\tcorrect\n')
        for (rel, t, lab), p in zip(rows, preds):
            f.write(f'{rel}\t{lab}\t{t}\t{p}\t{int(t == p)}\n')
    print(f'== {name} on {split} ({len(rows)} crops, {result["seconds_per_crop"] * 1000:.1f} ms/crop)')
    print(format_table(result))
    return result


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


def main():
    # dataset VietOCR giữ handle lmdb không pickle được → worker DataLoader phải fork
    if sys.platform.startswith('linux'):
        torch.multiprocessing.set_start_method('fork', force=True)
    ap = argparse.ArgumentParser()
    ap.add_argument('--data-root', required=True)
    ap.add_argument('--config', required=True)
    ap.add_argument('--work-dir', default='/kaggle/working/runs_vaipe')
    ap.add_argument('--iters', type=int, default=20000)
    ap.add_argument('--batch-size', type=int, default=32)
    ap.add_argument('--valid-every', type=int, default=2000)
    ap.add_argument('--max-width', type=int, default=1024)   # ~5 % vùng rộng hơn 512 px ở cao 32
    ap.add_argument('--lr', type=float, default=3e-4)
    ap.add_argument('--device', default='cuda:0')
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--num-workers', type=int, default=3)
    opt = ap.parse_args()

    data_root = Path(opt.data_root).resolve()
    for split in ('train', 'val', 'test'):
        print(f'{split}: {len(read_split(data_root, split))} crops')
    # thư mục sạch: VietOCR cache lmdb ở thư mục hiện tại và âm thầm dùng lại nếu đã có
    work = Path(opt.work_dir).resolve()
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    os.chdir(work)
    set_seed(opt.seed)

    config = Cfg.load_config_from_file(opt.config)
    config['device'] = opt.device
    config['cnn']['pretrained'] = False   # mọi trọng số lấy từ config['pretrain']
    config['dataset'].update({'name': 'pres_vaipe', 'data_root': str(data_root),
                              'train_annotation': 'train.txt', 'valid_annotation': 'val.txt',
                              'image_max_width': opt.max_width})
    config['trainer'].update({'iters': opt.iters, 'batch_size': opt.batch_size,
                              'print_every': min(200, opt.iters),
                              'valid_every': min(opt.valid_every, opt.iters),   # chỉ export ở các lần val
                              'metrics': 100000,                                 # dùng toàn bộ val
                              'export': str(work / 'weights' / 'transformerocr.pth'),
                              'checkpoint': str(work / 'checkpoint' / 'transformerocr_checkpoint.pth'),
                              'log': str(work / 'train.log')})
    config['optimizer']['max_lr'] = opt.lr
    config['dataloader']['num_workers'] = opt.num_workers
    config['seed'] = opt.seed

    # C2: pretrained, không fine-tune
    c2 = Cfg(dict(config))
    c2['weights'] = PRETRAINED_URL
    evaluate_split(c2, data_root, 'test', work, 'C2_pretrained')

    # C3: fine-tune trên train, chọn theo độ chính xác nguyên dòng trên val
    (work / 'weights').mkdir(exist_ok=True)
    (work / 'checkpoint').mkdir(exist_ok=True)
    Trainer(config, pretrained=True).train()
    weights = Path(config['trainer']['export'])
    if not weights.exists():
        raise SystemExit('Không export được trọng số (acc val không bao giờ > 0).')
    config['weights'] = str(weights)
    config.save(str(work / 'config.yml'))
    result = evaluate_split(config, data_root, 'test', work, 'C3_finetune')
    result.update({'seed': opt.seed, 'iters': opt.iters, 'batch_size': opt.batch_size,
                   'max_lr': opt.lr, 'selected_on': 'val exact-match accuracy'})
    json.dump(result, open(work / 'C3_finetune_metrics.json', 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
    a = result['all']
    print(f'TEST RESULT (VietOCR C3): crops={a["lines"]}  CER={a["cer"]:.4f}  WER={a["wer"]:.4f}  '
          f'acc={a["acc_full_sequence"]:.4f}')


if __name__ == '__main__':
    main()

In [ ]:
!mkdir -p {OUT_DIR}/logs
!CUDA_VISIBLE_DEVICES=0 python {WORK}/vietocr_ft.py --data-root {OCR_DIR} --config {WORK}/vgg_transformer.yml \
    --iters {ITERS} --valid-every {VALID_EVERY} --batch-size {BATCH} --max-width {MAX_WIDTH} --lr {LR} \
    --device cuda:0 --seed {SEED} 2>&1 | tee {OUT_DIR}/logs/vietocr.log \
    | grep -E "crops|valid loss|^==|^(class|all|drugname|quantity|usage|diagnose|date|other) |TEST RESULT|Error|Traceback"

## Gom kết quả

In [ ]:
RUN = Path(WORK) / 'runs_vaipe'
dst = Path(OUT_DIR) / 'vietocr'
dst.mkdir(parents=True, exist_ok=True)
shutil.copy(RUN / 'weights' / 'transformerocr.pth', dst / 'vietocr_vaipe.pth')
shutil.copy(RUN / 'config.yml', dst / 'vietocr_config.yml')
for f in [RUN / 'train.log', *RUN.glob('C*_metrics.json'), *RUN.glob('C*_predictions.tsv'), OCR_DIR / 'stats.json']:
    shutil.copy(f, dst)
!cd {OUT_DIR} && rm -f results.zip && zip -qr results.zip . -x results.zip && ls -la {OUT_DIR} {OUT_DIR}/vietocr
for name, ref in (('C2_pretrained', (0.0864, 0.1097, 0.8156)), ('C3_finetune', (0.0329, 0.0437, 0.9229))):
    a = json.load(open(dst / f'{name}_metrics.json'))['all']
    print(f"{name}: CER={a['cer']:.4f} WER={a['wer']:.4f} acc={a['acc_full_sequence']:.4f}  (gốc: {ref})")

**Dùng ở repo:** tải `output/results.zip`, đặt `vietocr/vietocr_vaipe.pth` → `code/weights/vietocr_finetuned.pth`;
metrics + predictions → `results/kaggle_vietocr/` cho bảng E-C1. Suy luận local dùng `code/configs/vietocr_vgg_transformer.yml`
(`image_max_width: 1024`, phải khớp lúc train).